# Training the network in Google Colab (GPU)

Before you start: **Runtime → Change runtime type → T4 GPU**.

At the end, download `emotion_cnn.pt` and put it in the `models/` folder of the project on your computer.

## 1. Upload the project
Choose the project zip file.

In [ ]:
from google.colab import files
up = files.upload()
zip_name = next(iter(up))
!unzip -q -o "$zip_name"
import glob, os
# the project folder: the one with src/gradcam.py (works for the project zip or the whole repository)
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(sorted(glob.glob('**/src/gradcam.py', recursive=True), key=len)[0]))))
print('Project folder:', os.getcwd())
!nvidia-smi -L

## 2. Install libraries
PyTorch is already installed in Colab.

In [ ]:
!pip install -q "opencv-python>=4.8,<5" kagglehub

## 3. Download FER-2013
If the download fails, upload `kaggle.json` (from your Kaggle account settings) and try again, or upload `archive.zip` manually.

In [ ]:
import kagglehub, shutil, os
path = kagglehub.dataset_download("msambare/fer2013")
print(path, os.listdir(path))
shutil.copytree(path, "data/fer2013", dirs_exist_ok=True)
!ls data/fer2013

## 4. Explore the data

In [ ]:
!python -m src.explore_data
from IPython.display import Image, display
for f in ["class_distribution", "samples_grid", "mean_faces"]:
    display(Image(f"reports/{f}.png"))

## 5. Train

In [ ]:
!python -m src.train --epochs 50 --workers 2

In [ ]:
display(Image("reports/training_curves.png"))

## 6. Evaluate on the test set

In [ ]:
!python -m src.evaluate
for f in ["confusion_matrix", "predictions_grid", "gradcam_grid"]:
    display(Image(f"reports/{f}.png"))

## 7. Download the model and reports

In [ ]:
!zip -q -r reports.zip reports
files.download("models/emotion_cnn.pt")
files.download("reports.zip")

## 8. (Optional) Export for the web page (GitHub Pages)
Packs the trained network into `docs/model/`, so the page in `docs/` can run it in the browser. Extract `docs_model.zip` into the project folder on your computer, then publish (see the README).

In [ ]:
!python -m src.export_web
!zip -q -r docs_model.zip docs/model
files.download('docs_model.zip')